# Sample creation Notebook
Make 2 samples per observation type for the following four scenario's
- PASS: Job samples with all passes (from STANDARD_STATUS)
- WARN: Job samples with passes and one or more upper or lower warnings triggered
- FAIL: Job samples containing upper or lower failures
- FAIL_IGN: Job samples containing upper or lower failures that were ignored

A job can be found by a unique combination of JOB_CODE, STD_CODE, STD_LOT_CODE and SCHEME_CODE.
naming scheme of samples: ANOMALY_[Sample Type]_x where x is 1 or 2

data to be saved in data\samples

filters from data\raw\ResultSet.csv: 
- BLANK - ANALYTICAL_TYPE == "Blank"
- CONTROL - ANALYTICAL_TYPE == "Standard" and STD_LOT_CODE not like substring "OREAS"
- SRMS - ANALYTICAL_TYPE == "Standard" and STD_LOT_CODE like substring "OREAS"
- DUP - ANALYTICAL_TYPE == "Replicate"
- REP - ANALYTICAL_TYPE == "Duplicate"
- MS - `SPK(MS) Assessment` sheet in `data\raw\QC_Anomaly_Training_Data_v2.xlsx`, where ANALYTICAL_TYPE == "Spike"

## load historic set

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd


df = pd.read_csv("../data/raw/ResultSet.csv")
df.columns = [c.strip().upper() for c in df.columns]

for col in ["NUMERIC_FINAL_VALUE", "INTERNAL_TARGET_VALUE",
            "INTERNAL_MAX_WARNING_VALUE", "INTERNAL_MIN_WARNING_VALUE",
            "INTERNAL_MAX_VALUE", "INTERNAL_MIN_VALUE"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["ANALYSED_DATE"] = pd.to_datetime(df["ANALYSED_DATE"], errors="coerce")

print(f"Loaded {len(df):,} rows")
print("Available ANALYTICAL_TYPE:")
print(df["ANALYTICAL_TYPE"].value_counts().to_string())

C:\Users\Jurriaan\AppData\Local\Temp\ipykernel_50572\648649206.py:8: DtypeWarning: Columns (0: INSTRUMENT_ID) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/raw/ResultSet.csv")


Loaded 99,999 rows
Available ANALYTICAL_TYPE:
ANALYTICAL_TYPE
Replicate    40240
Standard     36687
Blank        17997
Duplicate     5075


In [2]:
# change setting to show all columns in df.head()
pd.set_option('display.max_columns', None)
df[df['ANALYTICAL_TYPE'] == 'Blank'].head(10)

,ANALYTICAL_TYPE,STD_LOT_CODE,STD_CODE,JOB_CODE,NUMERIC_FINAL_VALUE,ANALYSED_DATE,SCHEME_CODE,ANALYTE_CODE,STANDARD_STATUS,PRECISION_STATUS,INTERNAL_MIN_VALUE,INTERNAL_MAX_VALUE,INTERNAL_MIN_INCLUSIVE,INTERNAL_MAX_INCLUSIVE,INTERNAL_MAX_WARNING_VALUE,INTERNAL_MIN_WARNING_VALUE,INTERNAL_MIN_WARNING_INCLUSIVE,INTERNAL_MAX_WARNING_INCLUSIVE,LIM_REP_VALUE,STAT_DL_VALUE,LIM_REP_DUP_VALUE,STAT_DL_DUP_VALUE,INTERNAL_TARGET_VALUE,PARENT_NUMERIC_FINAL_VALUE,UNIT_CODE,SPECIFICATION_CODE,INSTRUMENT_ID
6,Blank,TSV_BLANK,TSV_BLANK,TSV_LB0016663011,-0.139828,2021-11-22 16:36:25,GE_IMS40Q12,NI,Pass,NaN,-5.000,5.000,Y,Y,NaN,NaN,Y,Y,10,5.000,15.0,5.000,0.0,NaN,MG_KG,TSV_BLANK,NaN
16,Blank,TSV_BLANK,TSV_BLANK,TSV_LB0016663011,0.241648,2021-11-22 16:35:30,GE_IMS40Q12,CU,Pass,NaN,-5.000,5.000,Y,Y,NaN,NaN,Y,Y,10,5.000,15.0,5.000,0.0,NaN,MG_KG,TSV_BLANK,NaN
17,Blank,TSV_BLANK,TSV_BLANK,TSV_LB0016663011,0.335467,2021-11-22 16:35:19,GE_IMS40Q12,ZN,Pass,NaN,-12.500,12.500,Y,Y,NaN,NaN,Y,Y,10,12.500,15.0,12.500,0.0,NaN,MG_KG,TSV_BLANK,NaN
18,Blank,TSV_BLANK,TSV_BLANK,TSV_LB0016663011,-0.034207,2021-11-22 16:35:09,GE_IMS40Q12,PB,Pass,NaN,-2.500,2.500,Y,Y,NaN,NaN,Y,Y,10,1.250,15.0,1.250,0.0,NaN,MG_KG,TSV_BLANK,NaN
19,Blank,TSV_BLANK,TSV_BLANK,TSV_LB0016663011,0.011002,2021-11-22 16:34:53,GE_IMS40Q12,AG,Pass,NaN,-0.125,0.125,Y,Y,NaN,NaN,Y,Y,10,0.050,15.0,0.050,0.0,NaN,MG_KG,TSV_BLANK,NaN
22,Blank,TSV_BLANK,TSV_BLANK,TSV_LB0015957216,0.010400,2021-11-10 07:12:48,GE_IMS40Q12,BE,Pass,NaN,-0.250,0.250,Y,Y,NaN,NaN,Y,Y,10,0.250,15.0,0.250,0.0,NaN,MG_KG,TSV_BLANK,NaN
28,Blank,TSV_BLANK,TSV_BLANK,TSV_LB0015957216,0.209100,2021-11-10 07:12:38,GE_IMS40Q12,MO,Pass,NaN,-0.250,0.250,Y,Y,NaN,NaN,Y,Y,10,0.125,15.0,0.125,0.0,NaN,MG_KG,TSV_BLANK,NaN
33,Blank,TSV_BLANK,TSV_BLANK,TSV_LB0015957216,0.118750,2021-11-10 07:11:51,GE_IMS40Q12,SB,Pass,NaN,-0.250,0.250,Y,Y,NaN,NaN,Y,Y,10,0.125,15.0,0.125,0.0,NaN,MG_KG,TSV_BLANK,NaN
36,Blank,TSV_BLANK,TSV_BLANK,TSV_LB0015957216,0.223000,2021-11-10 07:11:35,GE_IMS40Q12,SE,Pass,NaN,-5.000,5.000,Y,Y,NaN,NaN,Y,Y,10,5.000,15.0,5.000,0.0,NaN,MG_KG,TSV_BLANK,NaN
41,Blank,TSV_BLANK,TSV_BLANK,TSV_LB0015957216,0.002700,2021-11-10 07:11:28,GE_IMS40Q12,U,Pass,NaN,-0.125,0.125,Y,Y,NaN,NaN,Y,Y,10,0.125,15.0,0.125,0.0,NaN,MG_KG,TSV_BLANK,NaN


In [3]:
df[df['ANALYTICAL_TYPE'] == 'Blank'].info()

<class 'pandas.DataFrame'>
Index: 17997 entries, 6 to 99998
Data columns (total 27 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   ANALYTICAL_TYPE                 17997 non-null  str           
 1   STD_LOT_CODE                    17997 non-null  str           
 2   STD_CODE                        17997 non-null  str           
 3   JOB_CODE                        17997 non-null  str           
 4   NUMERIC_FINAL_VALUE             17997 non-null  float64       
 5   ANALYSED_DATE                   17997 non-null  datetime64[us]
 6   SCHEME_CODE                     17997 non-null  str           
 7   ANALYTE_CODE                    17831 non-null  str           
 8   STANDARD_STATUS                 17993 non-null  str           
 9   PRECISION_STATUS                0 non-null      str           
 10  INTERNAL_MIN_VALUE              17997 non-null  float64       
 11  INTERNAL_MAX_VALUE

In [4]:
JOB_COLS = ["JOB_CODE", "STD_CODE", "STD_LOT_CODE", "SCHEME_CODE"]
SCENARIOS = ["PASS", "WARN", "FAIL", "FAIL_IGN"]
SAMPLES_DIR = Path("../data/samples")
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)

log_entries = []  # (filename, n_rows) for created samples
skipped = []      # (type_name, scenario) for scenarios with no available job


def classify_job(status_series):
    """Classify a job's rows into PASS/WARN/FAIL/FAIL_IGN from a status column."""
    s = status_series.dropna().astype(str)
    if s.empty:
        return None
    has_fail = s.str.contains("Failure") & ~s.str.contains("Ignor")
    has_fail_ign = s.str.contains("Failure") & s.str.contains("Ignor")
    has_warn = s.str.contains("Warning") & ~s.str.contains("Ignor")
    if has_fail.any():
        return "FAIL"
    if has_fail_ign.any():
        return "FAIL_IGN"
    if has_warn.any():
        return "WARN"
    return "PASS"


def build_scenario_jobs(df_filtered, status_col):
    """Return dict scenario -> sorted list of job-key tuples."""
    job_status = (
        df_filtered.groupby(JOB_COLS)[status_col]
        .apply(classify_job)
        .dropna()
    )
    out = {sc: [] for sc in SCENARIOS}
    for job_key, scenario in job_status.items():
        out[scenario].append(job_key)
    for sc in out:
        out[sc] = sorted(out[sc])
    return out


def save_job_samples(df_filtered, type_name, status_col):
    """Save up to 2 job samples per scenario for one observation type."""
    scenario_jobs = build_scenario_jobs(df_filtered, status_col)
    for scenario in SCENARIOS:
        job_keys = scenario_jobs[scenario][:2]
        if not job_keys:
            skipped.append((type_name, scenario))
            continue
        for i, job_key in enumerate(job_keys, start=1):
            mask = pd.Series(True, index=df_filtered.index)
            for col, val in zip(JOB_COLS, job_key):
                mask &= df_filtered[col] == val
            sample_df = df_filtered[mask]
            filename = f"{type_name}_{scenario}_{i}.csv"
            sample_df.to_csv(SAMPLES_DIR / filename, index=False)
            log_entries.append((filename, len(sample_df)))

## Blank Sample 

In [5]:
df_blank = df[df["ANALYTICAL_TYPE"] == "Blank"]
save_job_samples(df_blank, "BLANK", "STANDARD_STATUS")

## Control sample 

In [6]:
df_control = df[
    (df["ANALYTICAL_TYPE"] == "Standard")
    & ~df["STD_LOT_CODE"].str.contains("OREAS", na=False)
]
save_job_samples(df_control, "CONTROL", "STANDARD_STATUS")

## Duplicate sample

In [7]:
df_dup = df[df["ANALYTICAL_TYPE"] == "Duplicate"]
save_job_samples(df_dup, "DUP", "PRECISION_STATUS")

df_rep = df[df["ANALYTICAL_TYPE"] == "Replicate"]
save_job_samples(df_rep, "REP", "PRECISION_STATUS")

## SRMS sample

In [8]:
df_srms = df[
    (df["ANALYTICAL_TYPE"] == "Standard")
    & df["STD_LOT_CODE"].str.contains("OREAS", na=False)
]
save_job_samples(df_srms, "SRMS", "STANDARD_STATUS")

## Matrix Spike sample

Matrix Spike records come from the supplementary workbook. The redundant source-only `QC_TYPE` column is excluded because `ANALYTICAL_TYPE = Spike` already identifies Matrix Spike samples.

In [9]:
MATRIX_SPIKE_PATH = Path("../data/raw/QC_Anomaly_Training_Data_v2.xlsx")
MATRIX_SPIKE_SHEET = "SPK(MS) Assessment"

df_ms = pd.read_excel(
    MATRIX_SPIKE_PATH,
    sheet_name=MATRIX_SPIKE_SHEET,
    engine="openpyxl",
)
df_ms.columns = [str(c).strip().upper() for c in df_ms.columns]

required_ms_columns = set(JOB_COLS + ["ANALYTICAL_TYPE", "STANDARD_STATUS"])
missing_ms_columns = sorted(required_ms_columns - set(df_ms.columns))
if missing_ms_columns:
    raise ValueError(f"Matrix Spike sheet is missing required columns: {missing_ms_columns}")

df_ms = df_ms[
    df_ms["ANALYTICAL_TYPE"].astype("string").str.strip().str.casefold().eq("spike")
].copy()
if df_ms.empty:
    raise ValueError("No Matrix Spike rows matched ANALYTICAL_TYPE='Spike'.")
df_ms = df_ms.drop(columns="QC_TYPE", errors="ignore")

for col in ["NUMERIC_FINAL_VALUE", "INTERNAL_TARGET_VALUE",
            "INTERNAL_MAX_WARNING_VALUE", "INTERNAL_MIN_WARNING_VALUE",
            "INTERNAL_MAX_VALUE", "INTERNAL_MIN_VALUE"]:
    df_ms[col] = pd.to_numeric(df_ms[col], errors="coerce")
df_ms["ANALYSED_DATE"] = pd.to_datetime(df_ms["ANALYSED_DATE"], errors="coerce")

print(f"Loaded {len(df_ms):,} Matrix Spike rows from {MATRIX_SPIKE_SHEET!r}")
save_job_samples(df_ms, "MS", "STANDARD_STATUS")

Loaded 6,958 Matrix Spike rows from 'SPK(MS) Assessment'


## Sample column structure check

In [10]:
existing_sample_paths = sorted(
    path for path in SAMPLES_DIR.glob("*.csv")
    if not path.name.startswith("MS_")
)
matrix_spike_sample_paths = sorted(SAMPLES_DIR.glob("MS_*.csv"))

if not existing_sample_paths or not matrix_spike_sample_paths:
    raise ValueError("Both existing and Matrix Spike samples are required for the schema check.")

reference_path = existing_sample_paths[0]
reference_columns = pd.read_csv(reference_path, nrows=0).columns.tolist()
existing_schemas_match = all(
    pd.read_csv(path, nrows=0).columns.tolist() == reference_columns
    for path in existing_sample_paths
)
matrix_spike_schemas = {
    path.name: pd.read_csv(path, nrows=0).columns.tolist()
    for path in matrix_spike_sample_paths
}
matrix_spike_schemas_match = all(
    columns == reference_columns
    for columns in matrix_spike_schemas.values()
)

print(f"Reference schema: {reference_path.name} ({len(reference_columns)} columns)")
print(f"Existing sample schemas consistent: {existing_schemas_match}")
print(f"Matrix Spike schema matches existing samples: {matrix_spike_schemas_match}")

if not matrix_spike_schemas_match:
    ms_columns = next(iter(matrix_spike_schemas.values()))
    print("Extra Matrix Spike columns:", [c for c in ms_columns if c not in reference_columns])
    print("Missing Matrix Spike columns:", [c for c in reference_columns if c not in ms_columns])
    print(
        "Shared columns retain existing order:",
        [c for c in ms_columns if c in reference_columns] == reference_columns,
    )

Reference schema: BLANK_FAIL_1.csv (27 columns)
Existing sample schemas consistent: True
Matrix Spike schema matches existing samples: True


## Generation log

In [11]:
from datetime import datetime

log_path = SAMPLES_DIR / "sample_generation_log.txt"
generated_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
author = "JurriaanvdS <jjvanderstruijk@hotmail.com>"

lines = ["Test sample generation log", f"Generated: {generated_at}", f"By: {author}", ""]
for filename, n_rows in log_entries:
    lines.append(f"{filename}: {n_rows} records")
if skipped:
    lines.append("")
    lines.append("Skipped (no matching jobs found):")
    for type_name, scenario in skipped:
        lines.append(f"{type_name}_{scenario}: skipped")

log_path.write_text("\n".join(lines) + "\n")
print(f"Wrote {len(log_entries)} sample file(s) and log to {log_path}")

Wrote 46 sample file(s) and log to ..\data\samples\sample_generation_log.txt
